# Cutting WUSTL for Ethiopia

Downloaded annual means (GL) for 202301-202312: [Link](https://sites.wustl.edu/acag/surface-pm2-5/#V6.GL.02.04)

In [2]:
import xarray as xr
import geopandas as gpd

In [5]:
# Open the downloaded NetCDF file
ds = xr.open_dataset('V6GL02.04.CNNPM25.0p10.GL.202301-202312.nc', chunks={})
ds

<xarray.Dataset> Size: 19MB
Dimensions:  (lat: 1300, lon: 3600)
Coordinates:
  * lat      (lat) float32 5kB -59.95 -59.85 -59.75 -59.65 ... 69.75 69.85 69.95
  * lon      (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.8 179.9 179.9
Data variables:
    PM25     (lat, lon) float32 19MB dask.array<chunksize=(650, 1800), meta=np.ndarray>
Attributes:
    TITLE:            Convolutional Neural Network Annual PM2.5 Estimation ov...
    CONTACT:          SIYUAN SHEN <s.siyuan@wustl.edu>
    LAT_DELTA:        0.1
    LON_DELTA:        0.1
    SPATIALCOVERAGE:  GL
    TIMECOVERAGE:     2023

In [15]:
ds.rio.to_raster('gl_wustl_2023.tif')

In [14]:
ds['PM25']

<xarray.DataArray 'PM25' (lat: 1300, lon: 3600)> Size: 19MB
dask.array<open_dataset-PM25, shape=(1300, 3600), dtype=float32, chunksize=(650, 1800), chunktype=numpy.ndarray>
Coordinates:
  * lat          (lat) float32 5kB -59.95 -59.85 -59.75 ... 69.75 69.85 69.95
  * lon          (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.8 179.9 179.9
    spatial_ref  int64 8B 0
Attributes:
    units:      ug/m3
    long_name:  Convolutional Neural Network derived Annual PM2.5 [ug/m^3]

In [7]:
eth_admin = gpd.read_file('eth_admin0.geojson')

In [8]:
# We assign the CRS explicitly
ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
ds = ds.rio.write_crs("EPSG:4326")

In [9]:
# Ensure that both Delhi file and Netcdf file has same CRS
eth_admin = eth_admin.to_crs(ds.rio.crs)

In [10]:
# Get the bounding box coordinates of Eth
minx, miny, maxx, maxy = eth_admin.total_bounds

#Crop the netcdf to the bounding box
clipped = ds.rio.clip_box(
    minx=minx,
    miny=miny,
    maxx=maxx,
    maxy=maxy
)

In [11]:
pm25 = clipped['PM25']
pm25

<xarray.DataArray 'PM25' (lat: 115, lon: 151)> Size: 69kB
dask.array<getitem, shape=(115, 151), dtype=float32, chunksize=(99, 151), chunktype=numpy.ndarray>
Coordinates:
  * lat          (lat) float32 460B 3.45 3.55 3.65 3.75 ... 14.65 14.75 14.85
  * lon          (lon) float32 604B 32.95 33.05 33.15 ... 47.75 47.85 47.95
    spatial_ref  int64 8B 0
Attributes:
    units:      ug/m3
    long_name:  Convolutional Neural Network derived Annual PM2.5 [ug/m^3]

In [12]:
pm25.rio.to_raster('meanpm25_ethiopia_wustl_2023.tif')

In [6]:
stations = gpd.read_file('stations.geojson')

# Generate 200 random values from a standard normal distribution
random_values = np.random.normal(loc=50.0, scale=10.0, size=200)
stations['value'] = random_values

stations

,id,geometry,value
0,0,POINT (77.39685 13.02116),59.759894
1,1,POINT (77.38451 13.05559),44.905596
2,2,POINT (77.49955 12.96888),51.625487
3,3,POINT (77.47845 12.91619),55.341872
4,4,POINT (77.56241 13.10942),49.096248
...,...,...,...
195,195,POINT (77.45916 13.0201),50.086897
196,196,POINT (77.71024 12.95226),56.666848
197,197,POINT (77.61741 13.15991),38.158577
198,198,POINT (77.4864 13.15997),47.678120


In [7]:
stations.to_file('stations.geojson')

In [8]:
stations = gpd.read_file('stations.geojson')


In [9]:
stations

,id,value,geometry
0,0,59.759894,POINT (77.39685 13.02116)
1,1,44.905596,POINT (77.38451 13.05559)
2,2,51.625487,POINT (77.49955 12.96888)
3,3,55.341872,POINT (77.47845 12.91619)
4,4,49.096248,POINT (77.56241 13.10942)
...,...,...,...
195,195,50.086897,POINT (77.45916 13.0201)
196,196,56.666848,POINT (77.71024 12.95226)
197,197,38.158577,POINT (77.61741 13.15991)
198,198,47.678120,POINT (77.4864 13.15997)
